In [ ]:
import geopandas as gpd

geo = gpd.read_file('./data/2020_Census_Tracts_20260318.geojson')
geo.head()

In [ ]:
import requests
import pandas as pd

url = (
    "https://api.census.gov/data/2020/dec/pl"
    "?get=P1_001N,NAME"
    "&for=tract:*"
    "&in=state:36+county:005,047,061,081,085"
)

r = requests.get(url)
data = r.json()

pop = pd.DataFrame(data[1:], columns=data[0])

# Build the standard 11-digit GEOID to match your GeoJSON
pop['geoid'] = pop['state'] + pop['county'] + pop['tract']
pop['population'] = pop['P1_001N'].astype(int)

pop = pop[['geoid', 'population']]
print(pop.shape)
pop.head()

In [ ]:
gdf = geo.merge(pop, on='geoid', how='left')

print(f"Tracts total: {len(gdf)}")
print(f"Tracts missing population: {gdf['population'].isna().sum()}")
gdf[['geoid', 'boroname', 'ctlabel', 'population', 'geometry']].head()

In [ ]:
gdf['population'].sum() # Matches nyc population!

In [ ]:
gdf.to_file('./data/nyc_tract_population.geojson', driver='GeoJSON')

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

gdf.plot(
    column='population',
    ax=ax,
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Population', 'shrink': 0.6}
)

ax.set_title('NYC Census Tract Population (2020)', fontsize=16)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## We have census data by tract now! Lets try to map to H3!

In [ ]:
!pip install h3 h3pandas
import h3pandas

In [ ]:
gdf_proj = gdf.to_crs(epsg=2263)  # NY State Plane, feet

In [ ]:
import h3
import geopandas as gpd
from shapely.geometry import Polygon

nyc_wgs84 = gdf.union_all().convex_hull
cells = h3.geo_to_cells(nyc_wgs84.__geo_interface__, res=8)

# h3.cell_to_boundary returns (lat, lng) — must flip to (lng, lat) for shapely
hex_geoms = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells]
gdf_hex = gpd.GeoDataFrame({'h3_index': list(cells), 'geometry': hex_geoms}, crs='EPSG:4326')
gdf_hex = gdf_hex.to_crs(epsg=2263)

print(f"Total H3 cells: {len(gdf_hex)}")
gdf_hex.plot()  # quick visual sanity check — should look like NYC

In [ ]:
# Intersection gives us every hex-tract fragment with its geometry
intersected = gdf_hex.overlay(gdf_proj[['geoid', 'population', 'geometry']], how='intersection')

# Compute area of each fragment
intersected['fragment_area'] = intersected.geometry.area

# Compute total area of the original tract
tract_areas = gdf_proj[['geoid', 'geometry']].copy()
tract_areas['tract_area'] = tract_areas.geometry.area
intersected = intersected.merge(tract_areas[['geoid', 'tract_area']], on='geoid')

# Area-weighted population for each fragment
intersected['pop_fragment'] = intersected['population'] * (intersected['fragment_area'] / intersected['tract_area'])

In [ ]:
gdf_h3_pop = (
    intersected
    .groupby('h3_index')['pop_fragment']
    .sum()
    .reset_index()
    .rename(columns={'pop_fragment': 'population'})
)

# Join back to hex geometries
gdf_h3_pop = gdf_hex.merge(gdf_h3_pop, on='h3_index', how='left')
gdf_h3_pop['population'] = gdf_h3_pop['population'].fillna(0)

print(f"Total population across hexes: {gdf_h3_pop['population'].sum():,.0f}")
print(gdf_h3_pop.head())

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
gdf_h3_pop.plot(column='population', ax=ax, cmap='YlOrRd', legend=True,
                legend_kwds={'label': 'Population', 'shrink': 0.6})
ax.set_title('NYC Population by H3 Cell (Res 8, 2020)', fontsize=16)
ax.set_axis_off()
plt.tight_layout()
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Remove zero-pop cells (water/boundary)
gdf_plot = gdf_h3_pop[gdf_h3_pop['population'] > 0].copy()

# Log scale population
gdf_plot['pop_log'] = np.log1p(gdf_plot['population'])

fig, ax = plt.subplots(1, 1, figsize=(10, 12))

gdf_plot.plot(
    column='pop_log',
    ax=ax,
    cmap='Blues',
    legend=True,
    legend_kwds={'label': 'Population (log scale)', 'shrink': 0.6}
)

ax.set_title('NYC Population by H3 Cell (Res 8, 2020)', fontsize=16)
ax.set_axis_off()
plt.tight_layout()
# plt.show()

In [ ]:
import folium
import json

# Convert to WGS84 for folium
gdf_plot = gdf_h3_pop[gdf_h3_pop['population'] > 0].copy()
gdf_plot = gdf_plot.to_crs(epsg=4326)

# Normalize population for color scaling
import branca.colormap as cm
colormap = cm.linear.YlOrRd_09.scale(gdf_plot['population'].min(), gdf_plot['population'].max())

m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB positron')

for _, row in gdf_plot.iterrows():
    folium.GeoJson(
        row['geometry'].__geo_interface__,
        style_function=lambda x, pop=row['population']: {
            'fillColor': colormap(pop),
            'color': 'none',
            'fillOpacity': 0.7,
        }
    ).add_to(m)

colormap.caption = 'Population per H3 Cell'
colormap.add_to(m)

m.save('./data/nyc_h3_population.html')
m

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

gdf.plot(
    column='population',
    ax=ax,
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Population per Census Tract', 'shrink': 0.6}
)

ax.set_title('NYC Population by Census Tract (2020)', fontsize=16)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('./data/nyc_tract_population_map.png', dpi=150, bbox_inches='tight')
plt.show()

## Summarized

In [ ]:
import geopandas as gpd
import pandas as pd
import requests, h3pandas, h3
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import folium
import branca.colormap as cm

# Read govt file
geo = gpd.read_file('./data/2020_Census_Tracts_20260318.geojson')

# Fetch pop data for NYC from census API
url = (
    "https://api.census.gov/data/2020/dec/pl"
    "?get=P1_001N,NAME"
    "&for=tract:*"
    "&in=state:36+county:005,047,061,081,085"
)

r = requests.get(url)
data = r.json()

pop = pd.DataFrame(data[1:], columns=data[0])

# Make pop DF to map Tract GeoID <-> population
pop['geoid'] = pop['state'] + pop['county'] + pop['tract']
pop['population'] = pop['P1_001N'].astype(int)

pop = pop[['geoid', 'population']]

# Merge pop & tract geometry
gdf = geo.merge(pop, on='geoid', how='left')

print(f"Tracts total: {len(gdf)}")
print(f"Tracts missing population: {gdf['population'].isna().sum()}")
# gdf[['geoid', 'boroname', 'ctlabel', 'population', 'geometry']].head()

gdf.to_file('./data/nyc_tract_population.geojson', driver='GeoJSON')


# Get NY State Plane in feet
gdf_proj = gdf.to_crs(epsg=2263) 


# Bound area to limit num of cells we calculate for
nyc_wgs84 = gdf.union_all().convex_hull
cells = h3.geo_to_cells(nyc_wgs84.__geo_interface__, res=8)

# h3.cell_to_boundary returns (lat, lng) — must flip to (lng, lat) for shapely
hex_geoms = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells]
gdf_hex = gpd.GeoDataFrame({'h3_index': list(cells), 'geometry': hex_geoms}, crs='EPSG:4326')
gdf_hex = gdf_hex.to_crs(epsg=2263)

print(f"Total H3 cells: {len(gdf_hex)}")
# gdf_hex.plot()

### Intersection Logic ###

# Intersection gives us every hex-tract fragment with its geometry
intersected = gdf_hex.overlay(gdf_proj[['geoid', 'population', 'geometry']], how='intersection')

# Compute area of each fragment
intersected['fragment_area'] = intersected.geometry.area

# Compute total area of the original tract
tract_areas = gdf_proj[['geoid', 'geometry']].copy()
tract_areas['tract_area'] = tract_areas.geometry.area
intersected = intersected.merge(tract_areas[['geoid', 'tract_area']], on='geoid')

# Area-weighted population for each fragment
intersected['pop_fragment'] = intersected['population'] * (intersected['fragment_area'] / intersected['tract_area'])

gdf_h3_pop = (
    intersected
    .groupby('h3_index')['pop_fragment']
    .sum()
    .reset_index()
    .rename(columns={'pop_fragment': 'population'})
)

# Join back to hex geometries
gdf_h3_pop = gdf_hex.merge(gdf_h3_pop, on='h3_index', how='left')
gdf_h3_pop['population'] = gdf_h3_pop['population'].fillna(0)
out = gdf_h3_pop[gdf_h3_pop['population'] > 0][['h3_index','population']]

print(f"Total population across hexes: {out['population'].sum():,.0f}")
out.head()


In [ ]:
import geopandas as gpd
import pandas as pd
import requests, h3pandas, h3, os
from shapely.geometry import Polygon
from supabase import create_client, Client
from dotenv import load_dotenv


load_dotenv() 

URL = os.environ.get("SUPABASE_URL")
KEY = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(URL, KEY)

def push_to_supabase(df, table_name):
    data = df.to_dict(orient='records')
    try:
        response = supabase.table(table_name).upsert(
            data,
            on_conflict="h3_index"
        ).execute()
        print(f"  Successfully pushed {len(data)} rows to {table_name}!")
        return response
    except Exception as e:
        print(f"  Error pushing to Supabase: {e}")

# Read govt file
print("Loading census tract geometries...")
geo = gpd.read_file('./data/2020_Census_Tracts_20260318.geojson')

# Fetch pop data for NYC from census API
print("\nFetching population data from Census API...")
url = (
    "https://api.census.gov/data/2020/dec/pl"
    "?get=P1_001N,NAME"
    "&for=tract:*"
    "&in=state:36+county:005,047,061,081,085"
)

r = requests.get(url)
data = r.json()

pop = pd.DataFrame(data[1:], columns=data[0])

# Make pop DF to map Tract GeoID <-> population
pop['geoid'] = pop['state'] + pop['county'] + pop['tract']
pop['population'] = pop['P1_001N'].astype(int)

pop = pop[['geoid', 'population']]

# Merge pop & tract geometry
gdf = geo.merge(pop, on='geoid', how='left')

# === Prep for intersection on all resolutions === #

# Get NY State Plane in feet
gdf_proj = gdf.to_crs(epsg=2263) 

# Bound area to limit num of cells we calculate for
nyc_wgs84 = gdf.union_all().convex_hull

tract_areas = gdf_proj[['geoid', 'geometry']].copy()
tract_areas['tract_area'] = tract_areas.geometry.area

# === Calculate pop for res 7, 8, 9 === #

for res in [7, 8, 9]:
    print(f"\nProcessing H3 resolution {res}...")
    
    print(f"  Generating H3 cells...")
    cells = h3.geo_to_cells(nyc_wgs84.__geo_interface__, res=res)

    hex_geoms = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells]
    gdf_hex = gpd.GeoDataFrame({'h3_index': list(cells), 'geometry': hex_geoms}, crs='EPSG:4326')
    gdf_hex = gdf_hex.to_crs(epsg=2263)
    print(f"  H3 cells generated: {len(gdf_hex)}")

    print(f"  Running intersection (this may take a moment)...")
    intersected = gdf_hex.overlay(gdf_proj[['geoid', 'population', 'geometry']], how='intersection')
    intersected['fragment_area'] = intersected.geometry.area
    intersected = intersected.merge(tract_areas[['geoid', 'tract_area']], on='geoid')
    intersected['pop_fragment'] = intersected['population'] * (intersected['fragment_area'] / intersected['tract_area'])

    gdf_h3_pop = (
        intersected
        .groupby('h3_index')['pop_fragment']
        .sum()
        .reset_index()
        .rename(columns={'pop_fragment': 'population'})
    )

    gdf_h3_pop = gdf_hex.merge(gdf_h3_pop, on='h3_index', how='left')
    gdf_h3_pop['population'] = gdf_h3_pop['population'].fillna(0)
    out = gdf_h3_pop[gdf_h3_pop['population'] > 0][['h3_index', 'population']]
    out['population'] = out['population'].round(1)

    print(f"  Total population: {out['population'].sum():,.0f} ({out['population'].sum() / 8_804_190 * 100:.2f}%)")
    print(f"  Total H3 cells: {out.shape[0]}")

    print("  Pushing to Supabase...")
    push_to_supabase(out, f"h3_population_res{res}")
    print("  Done!")

print("\nAll resolutions complete.")
